# Reflexion loop
This lets an AI 
  - Answer a question, 
  - Critique its own answer, and then 
  - Refine it—automatically. 

**This approach helps the model improve its responses through multiple rounds of self-evaluation.**

In [ ]:
import oci
from LoadProperties import LoadProperties
properties=LoadProperties()

# YOUR_COHERE_API_KEY = ""
# Initialize the Cohere LLM
# from langchain.llms import Cohere
# llm = Cohere(cohere_api_key=YOUR_COHERE_API_KEY, temperature=0.7)

# OCI Generative AI service LLM Access

from langchain_community.chat_models.oci_generative_ai import ChatOCIGenAI

llm = ChatOCIGenAI(
      model_id='meta.llama-3.3-70b-instruct',
      service_endpoint=properties.getEndpoint(),
      compartment_id=properties.getCompartment(),auth_type='INSTANCE_PRINCIPAL',
      model_kwargs={ "max_tokens": 1000},)

In [ ]:
# from langchain_cohere import ChatCohere
from langchain.prompts import PromptTemplate
from IPython.display import Markdown, display

# STEP 1: Initialize Cohere LLM
# cohere_api_key = "your_cohere_api_key"  # 🔑 Replace this with your Cohere API key
# llm = ChatCohere(cohere_api_key=cohere_api_key, model="command-r")

# STEP 2: Define Prompt Templates

# 2.1 == Answering the original question
answer_prompt = PromptTemplate(
    input_variables=["question"],
    template="You are a helpful assistant. Answer the question: {question}"
)

# 2.2 == Self-critique of the previous answer
critique_prompt = PromptTemplate(
    input_variables=["question", "answer"],
    template="""
You are an AI evaluator. Carefully critique the answer to the question.
- Identify inaccuracies, vagueness, or gaps.
- Suggest improvements.

Question: {question}
Answer: {answer}

Critique:
"""
)

# 2.3 == Refining the answer based on the critique
refine_prompt = PromptTemplate(
    input_variables=["question", "answer", "critique"],
    template="""
You are an assistant improving an answer based on critique.

Question: {question}
Previous Answer: {answer}
Critique: {critique}

Improved Answer:
"""
)

# STEP 3: Build LCEL pipelines by “piping” prompts into the LLM
answer_chain   = answer_prompt   | llm
critique_chain = critique_prompt | llm
refine_chain   = refine_prompt   | llm

# STEP 4:== Ask a Question
question = "Why do we see different constellations in different seasons?"
display(Markdown("**Question Asked : "+question+"**"))
print()

# STEP 5:== Reflexion Loop

# == Generate initial answer
current_answer = answer_chain.invoke({"question":question})
print(f"\n🔹 Initial Answer:\n")
display(Markdown(current_answer.content))

# Number of refinement rounds
num_iterations = 3

for i in range(num_iterations):
    print(f"\n🔁 Iteration {i+1}:")
    
    # == Critique the current answer
    critique = critique_chain.invoke({"question":question, "answer":current_answer})
    print(f"\n🧐 Critique:\n")
    display(Markdown(critique.content))
    
    # == Refine the answer based on the critique
    improved_answer = refine_chain.invoke({
        "question":question,
        "answer":current_answer,
        "critique":critique
})
    print(f"\n✅ Improved Answer:\n")
    display(Markdown(improved_answer.content))
    
    # Update current answer for the next iteration
    current_answer = improved_answer

# == Final output
print("\n🏁 Final Refined Answer After Iterations:")

# print(current_answer.content)
display(Markdown(current_answer.content))
